# Crash Severity Predictor & Cost-Weighted Hotspots — Fairfield County, CT
**Jan Salafia, Sean Salafia — Machine Learning final project**

Data: Connecticut Crash Data Repository (CTCDR), PR-1 era (2015 onward).

**Research question:** Which Fairfield County locations have the highest crash *severity*, and does a severity/cost-weighted hotspot ranking differ from a ranking by crash count? What conditions make crashes severe?

**Notebook outline**
1. Setup & configuration
2. Load data and auto-detect columns
3. Clean / filter (Fairfield County, 2015+) and build the severity target
4. Exploratory analysis: what conditions go with severe crashes?
5. **Severity predictor** (baseline logistic regression vs. gradient boosting, time-based split)
6. **Hotspots**: cluster crashes, rank by cost vs. by count, compare
7. What do the costliest hotspots have in common?
8. Map + exports

> Items marked **`# CHECK`** are things to verify against your actual CTCDR export.

## 1. Setup & configuration

In [ ]:
import re, glob, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.cluster import DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, classification_report, fbeta_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
RNG = 42

# ---------------- CONFIG ----------------
DATA_GLOB = "data/*.csv"          # CHECK: put your CTCDR CSV export(s) in ./data/ (several files are concatenated)
ALLOW_SYNTHETIC = True            # if no CSV is found, generate FAKE demo data so the notebook can be tested
START_YEAR = 2015

FAIRFIELD_COUNTY_TOWNS = [
    "Bethel","Bridgeport","Brookfield","Danbury","Darien","Easton","Fairfield","Greenwich","Monroe",
    "New Canaan","New Fairfield","Newtown","Norwalk","Redding","Ridgefield","Shelton","Sherman",
    "Stamford","Stratford","Trumbull","Weston","Westport","Wilton"]

# Severe = K (fatal) or A (suspected serious injury)  -> the "KSI" definition
SEVERE_CLASSES = {"K", "A"}

# Comprehensive crash cost per person-injury level (FHWA, 2016 dollars).
# CHECK: confirm against the FHWA Crash Cost tool/report you cite, and inflate to a common year if desired.
CRASH_COST = {"K": 11_295_400, "A": 655_000, "B": 198_500, "C": 125_600, "O": 11_900}

# Optional manual column overrides: {"severity": "Crash Severity", ...}. Leave empty to auto-detect.
MANUAL_MAP = {}

# Model features: auto-selected categorical/numeric columns; edit these to force include/exclude.
EXCLUDE_FEATURES = []
INCLUDE_FEATURES = []

# Hotspot clustering
EPS_METERS = 100        # crashes within this distance chain into one hotspot
MIN_CRASHES = 15        # minimum crashes to call a cluster a hotspot
TOP_N = 25              # size of the "top list" used when comparing rankings

## 2. Load data and auto-detect columns

The CTCDR export's exact column names can vary, so the code matches columns by keyword and prints what it found.
If something is wrong, set `MANUAL_MAP` above.

In [ ]:
def make_synthetic(n=60000, seed=0):
    """FAKE data with CTCDR-like columns. Only for testing the pipeline — results are meaningless."""
    rng = np.random.default_rng(seed)
    towns = {"Bridgeport":(41.1792,-73.1894,9),"Stamford":(41.0534,-73.5387,9),"Norwalk":(41.1177,-73.4082,7),
             "Danbury":(41.3948,-73.4540,6),"Fairfield":(41.1408,-73.2613,5),"Greenwich":(41.0262,-73.6282,4),
             "Stratford":(41.1845,-73.1332,4),"Shelton":(41.3165,-73.0932,3),"Trumbull":(41.2429,-73.2007,3),
             "Westport":(41.1415,-73.3579,3),"Bethel":(41.3715,-73.4140,2),"Brookfield":(41.4826,-73.4098,2),
             "Darien":(41.0787,-73.4693,2),"Easton":(41.2534,-73.2989,1),"Monroe":(41.3326,-73.2073,2),
             "New Canaan":(41.1465,-73.4948,2),"New Fairfield":(41.4676,-73.4890,1),"Newtown":(41.4140,-73.3040,2),
             "Redding":(41.3025,-73.3837,1),"Ridgefield":(41.2815,-73.4982,2),"Sherman":(41.5818,-73.4970,1),
             "Weston":(41.2007,-73.3812,1),"Wilton":(41.1954,-73.4379,2)}
    names = list(towns); w = np.array([towns[t][2] for t in names], float); w /= w.sum()
    town = rng.choice(names, n, p=w)
    base = np.array([towns[t][:2] for t in town])
    # 40% of crashes sit on ~60 "problem spots", the rest scatter around town centers
    spots = base[rng.choice(n, 60, replace=False)] + rng.normal(0, .004, (60, 2))
    spot_risk = rng.uniform(0.6, 2.2, 60)
    on_spot = rng.random(n) < 0.4
    sid = rng.integers(0, 60, n)
    ll = np.where(on_spot[:, None], spots[sid] + rng.normal(0, .0004, (n, 2)), base + rng.normal(0, .02, (n, 2)))
    risk = np.where(on_spot, spot_risk[sid], 1.0)
    dt = pd.to_datetime("2015-01-01") + pd.to_timedelta(rng.integers(0, 365*10 + 330, n), unit="D") \
         + pd.to_timedelta(rng.integers(0, 24, n), unit="h") \
         + pd.to_timedelta(rng.integers(0, 60, n), unit="m")
    light = rng.choice(["Daylight","Dark - Lighted","Dark - Not Lighted","Dusk","Dawn"], n, p=[.65,.18,.08,.06,.03])
    weather = rng.choice(["Clear","Rain","Snow","Cloudy","Fog"], n, p=[.6,.15,.05,.18,.02])
    surf = np.where(weather=="Snow", "Snow/Ice", np.where(weather=="Rain", "Wet", "Dry"))
    coll = rng.choice(["Rear-end","Angle","Sideswipe","Fixed object","Head-on","Pedestrian","Backing"], n,
                      p=[.30,.20,.12,.15,.03,.03,.17])
    route = rng.choice(["Local road","State route","Interstate","US route"], n, p=[.5,.25,.15,.10])
    speed = rng.choice([25,30,35,40,45,55,65], n, p=[.15,.25,.2,.15,.1,.1,.05])
    nveh = rng.choice([1,2,3,4], n, p=[.3,.55,.1,.05])
    factor = rng.choice(["None","Following too closely","Failed to grant right of way","Speeding","Distracted","Impaired","Unsafe lane change"],
                        n, p=[.2,.22,.18,.1,.15,.05,.10])
    z = (-4.3 + 0.015*(speed-35) + 0.6*(light=="Dark - Not Lighted") + 0.2*(light=="Dark - Lighted")
         + 1.3*(coll=="Pedestrian") + 1.0*(coll=="Head-on") + 0.5*(coll=="Fixed object") + 0.4*(coll=="Angle")
         - 0.6*(coll=="Backing") - 0.3*(coll=="Rear-end") + 0.5*(factor=="Impaired") + 0.4*(factor=="Speeding")
         + 0.3*(route=="Interstate") + np.log(risk))
    p = 1/(1+np.exp(-z))
    u = rng.random(n)
    sev = np.where(u < p*0.12, "K - Fatal Injury", np.where(u < p, "A - Suspected Serious Injury",
          np.where(u < p + .06, "B - Suspected Minor Injury", np.where(u < p + .15, "C - Possible Injury", "O - No Apparent Injury"))))
    df = pd.DataFrame({"Crash ID": np.arange(n), "Crash Date Time": dt.strftime("%m/%d/%Y %I:%M:%S %p"),
        "Town Name": town, "Latitude": ll[:, 0], "Longitude": ll[:, 1], "Crash Severity": sev,
        "Manner of Crash/Collision Impact": coll, "Light Condition": light, "Weather Condition": weather,
        "Road Surface Condition": surf, "Route Class": route, "Posted Speed Limit": speed,
        "Number of Vehicles": nveh, "Contributing Factor": factor,
        "Number of Injured": np.where(np.isin(sev, ["O - No Apparent Injury"]), 0, rng.integers(1, 3, n))})
    return df

files = sorted(glob.glob(DATA_GLOB))
USING_SYNTHETIC = False
if files:
    raw = pd.concat([pd.read_csv(f, low_memory=False).assign(_source_file=os.path.basename(f)) for f in files],
                    ignore_index=True)
    print(f"Loaded {len(files)} file(s), {len(raw):,} rows")
elif ALLOW_SYNTHETIC:
    raw = make_synthetic(); USING_SYNTHETIC = True
    print("!!! NO CSV FOUND — using FAKE synthetic data. Do not interpret any result. !!!")
else:
    raise FileNotFoundError(f"No files match {DATA_GLOB}")
print(raw.shape); raw.head(3)

In [ ]:
PATTERNS = {   # canonical name -> regexes tried in order against lower-cased column names
    "crash_id": [r"crash.*id", r"^id$", r"case"],
    "datetime": [r"crash.*date.*time", r"date.*time", r"datetime"],
    "date":     [r"^crash.*date$", r"^date$", r"crash.*date"],
    "time":     [r"^crash.*time$", r"^time$", r"crash.*time"],
    "town":     [r"^town", r"municipal", r"^city"],
    "lat":      [r"^lat"],
    "lon":      [r"^lon"],
    "severity": [r"crash.*severity", r"^severity", r"kabco", r"max.*injury"],
}
def detect_columns(df, manual):
    cols = {c.lower().strip(): c for c in df.columns}
    out = dict(manual)
    for key, pats in PATTERNS.items():
        if key in out: continue
        for p in pats:
            hit = [orig for low, orig in cols.items() if re.search(p, low)]
            if hit: out[key] = hit[0]; break
    return out

COL = detect_columns(raw, MANUAL_MAP)
for k in PATTERNS: print(f"{k:10s} -> {COL.get(k)}")
assert "severity" in COL and "lat" in COL and "lon" in COL and "town" in COL, "Set MANUAL_MAP for the missing columns"
assert "datetime" in COL or ("date" in COL), "Need a date (or datetime) column"

## 3. Clean, filter, and build the severity target

In [ ]:
def parse_kabco(s):
    """Map CTCDR severity text to K/A/B/C/O. Handles 'K - Fatal Injury', 'K', or plain descriptions."""
    if pd.isna(s): return np.nan
    t = str(s).strip().upper()
    m = re.match(r"^([KABCO])\b", t)
    if m: return m.group(1)
    if "FATAL" in t: return "K"
    if "SERIOUS" in t: return "A"
    if "MINOR" in t: return "B"
    if "POSSIBLE" in t: return "C"
    if "NO APPARENT" in t or "NO INJURY" in t or "PROPERTY" in t: return "O"
    return np.nan

df = raw.copy()
# --- datetime ---
if "datetime" in COL:
    df["dt"] = pd.to_datetime(df[COL["datetime"]], errors="coerce")
else:
    df["dt"] = pd.to_datetime(df[COL["date"]].astype(str) + " " + df[COL["time"]].astype(str) if "time" in COL
                              else df[COL["date"]], errors="coerce")
# --- location / town ---
df["town"] = df[COL["town"]].astype(str).str.strip().str.title()
df["lat"] = pd.to_numeric(df[COL["lat"]], errors="coerce")
df["lon"] = pd.to_numeric(df[COL["lon"]], errors="coerce")
# --- severity ---
df["kabco"] = df[COL["severity"]].map(parse_kabco)
print("Unparsed severity values:", df.loc[df.kabco.isna(), COL["severity"]].value_counts(dropna=False).head(8).to_dict())

n0 = len(df)
steps = {}
df = df[df.dt.notna()];                                   steps["valid date"] = len(df)
df = df[df.dt.dt.year >= START_YEAR];                     steps[f">= {START_YEAR}"] = len(df)
df = df[df.town.isin(FAIRFIELD_COUNTY_TOWNS)];            steps["Fairfield County"] = len(df)
df = df[df.kabco.notna()];                                steps["severity known"] = len(df)
# CT bounding box sanity check for coordinates; rows without coords stay for modelling but not for hotspots
bad_xy = ~(df.lat.between(40.9, 41.7) & df.lon.between(-73.8, -73.0))
df.loc[bad_xy, ["lat", "lon"]] = np.nan
print(pd.Series({"raw": n0, **steps}).to_frame("rows remaining"))
print(f"Missing/invalid coordinates: {df.lat.isna().mean():.1%}")

df["severe"] = df.kabco.isin(SEVERE_CLASSES).astype(int)
df["cost"] = df.kabco.map(CRASH_COST)
df["year"] = df.dt.dt.year
print(f"\nSevere (K+A) share: {df.severe.mean():.2%}   |  years {df.year.min()}–{df.year.max()}")
df.kabco.value_counts().reindex(list("KABCO")).to_frame("crashes").assign(
    share=lambda d: (d.crashes/d.crashes.sum()).round(4), unit_cost=lambda d: d.index.map(CRASH_COST))

In [ ]:
# Temporal features
df["hour"] = df.dt.dt.hour
df["dow"] = df.dt.dt.dayofweek
df["month"] = df.dt.dt.month
df["is_weekend"] = (df.dow >= 5).astype(int)
df["rush_hour"] = (df.hour.isin([7, 8, 16, 17, 18]) & (df.dow < 5)).astype(int)
df["late_night"] = df.hour.isin([22, 23, 0, 1, 2, 3, 4]).astype(int)

# Automatic feature selection from the remaining columns (printed so you can audit it).
LEAK = re.compile(r"sever|injur|kill|fatal|death|dead|transport|hospital|ambulance|\bems\b|cost|kabco|serious|casualt|"
                  r"medical|extric|^_source|^dt$|^cost$|^severe$|^year$", re.I)
USED_AS_KEY = set(COL.values())
auto_cat, auto_num = [], []
for c in raw.columns:
    if c in USED_AS_KEY or LEAK.search(c) or c in EXCLUDE_FEATURES or c not in df.columns: continue
    s = df[c]
    if s.isna().mean() > 0.5: continue
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
        if s.nunique() > 1: auto_num.append(c)
    elif pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
        if 2 <= s.nunique() <= 40: auto_cat.append(c)
cat_feats = [c for c in auto_cat + [f for f in INCLUDE_FEATURES if f in df.columns] if c not in EXCLUDE_FEATURES]
num_feats = [c for c in auto_num + ["hour", "dow", "month", "is_weekend", "rush_hour", "late_night"]]
cat_feats = list(dict.fromkeys(cat_feats)); num_feats = list(dict.fromkeys(num_feats))
print("Categorical features:", cat_feats)
print("Numeric features:   ", num_feats)
print("\nCHECK: remove anything that is only known *because* of the outcome (add to EXCLUDE_FEATURES).")

## 4. Exploratory analysis — what goes with severe crashes?
Severe-crash rate (K+A share) by condition. Look for categories with high rates **and** enough crashes to trust.

In [ ]:
def rate_table(col, min_n=200):
    t = df.groupby(col).agg(crashes=("severe", "size"), severe=("severe", "sum"), severe_rate=("severe", "mean")).reset_index()
    t["lift_vs_overall"] = t.severe_rate / df.severe.mean()
    return t[t.crashes >= min_n].sort_values("severe_rate", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (col, ttl) in zip(axes, [("hour", "Hour of day"), ("dow", "Day of week (0=Mon)"), ("year", "Year")]):
    t = df.groupby(col).severe.mean() * 100
    ax.bar(t.index, t.values, color="#2a6f97"); ax.set_title(f"Severe-crash rate by {ttl}"); ax.set_ylabel("% K+A")
plt.tight_layout(); plt.show()

for c in cat_feats[:8]:
    print(f"\n=== {c} ==="); display(rate_table(c).round(4))

## 5. Severity predictor
**Target:** `severe` = fatal (K) or suspected serious injury (A).  This is a rare-event problem (a few %), so accuracy is meaningless —
we judge models by **PR-AUC (average precision)**, ROC-AUC, and precision/recall at a chosen threshold.

**Split:** by *time*, not randomly (train on the past, test on the most recent two years) — this mimics real use and avoids leakage from near-duplicate neighbouring crashes.

In [ ]:
years = sorted(df.year.unique())
years = [int(y) for y in years]; test_years = years[-2:]; val_year = years[-3]
train = df[df.year < val_year]; val = df[df.year == val_year]; test = df[df.year.isin(test_years)]
print(f"train {len(train):,} (<{val_year}) | val {len(val):,} ({val_year}) | test {len(test):,} {test_years}")
print(f"severe rate  train {train.severe.mean():.2%} | val {val.severe.mean():.2%} | test {test.severe.mean():.2%}")

FEATS = cat_feats + num_feats
Xtr, ytr, Xva, yva, Xte, yte = train[FEATS], train.severe, val[FEATS], val.severe, test[FEATS], test.severe

# Baseline: logistic regression with one-hot categories
logit = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), cat_feats),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_feats)])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5))])

# Gradient boosting with native categorical handling
gb = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), cat_feats),
        ("num", "passthrough", num_feats)])),
    ("clf", HistGradientBoostingClassifier(
        categorical_features=list(range(len(cat_feats))), class_weight="balanced",
        learning_rate=0.06, max_depth=5, max_iter=400, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.15, n_iter_no_change=25, random_state=RNG))])

models = {"Logistic regression": logit, "Gradient boosting": gb}
for name, m in models.items(): m.fit(Xtr, ytr)

def best_threshold(y, p, beta=1.0):
    pr, rc, th = precision_recall_curve(y, p)
    f = (1 + beta**2) * pr[:-1] * rc[:-1] / np.maximum(beta**2 * pr[:-1] + rc[:-1], 1e-12)
    return th[np.argmax(f)]

rows, probs = [], {}
base_rate = yte.mean()
for name, m in models.items():
    thr = best_threshold(yva, m.predict_proba(Xva)[:, 1], beta=2)   # beta=2 favors recall (missing a severe crash is costly)
    p = m.predict_proba(Xte)[:, 1]; probs[name] = p; pred = p >= thr
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    rows.append({"model": name, "PR-AUC": average_precision_score(yte, p), "ROC-AUC": roc_auc_score(yte, p),
                 "threshold (val, F2)": thr, "precision": tp/max(tp+fp, 1), "recall": tp/max(tp+fn, 1),
                 "F2": fbeta_score(yte, pred, beta=2)})
res = pd.DataFrame(rows).set_index("model")
print(f"Test-set base rate (PR-AUC of a coin-flip ≈ this): {base_rate:.3f}")
res.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
for name, p in probs.items():
    PrecisionRecallDisplay.from_predictions(yte, p, name=name, ax=ax[0])
    RocCurveDisplay.from_predictions(yte, p, name=name, ax=ax[1])
ax[0].axhline(base_rate, ls="--", c="grey", lw=1); ax[0].set_title("Precision–Recall (test)"); ax[1].set_title("ROC (test)")
plt.tight_layout(); plt.show()

# Calibration of the better model: do predicted risks mean what they say?
best = res["PR-AUC"].idxmax(); p = probs[best]
q = pd.qcut(p, 10, duplicates="drop")
cal = pd.DataFrame({"p": p, "y": yte.values}).groupby(q).mean()
print(f"Best model: {best}. Predicted vs. observed severe rate by risk decile (balanced class weights inflate predicted p; "
      "ranking, not absolute value, is what matters):")
cal.round(4)

In [ ]:
# What drives predictions? Permutation importance on the test set (drop in PR-AUC when a feature is shuffled)
m = models[best]
samp = test.sample(min(20000, len(test)), random_state=RNG)
pi = permutation_importance(m, samp[FEATS], samp.severe, scoring="average_precision",
                            n_repeats=5, random_state=RNG, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values()
err = pd.Series(pi.importances_std, index=FEATS)[imp.index]
fig, ax = plt.subplots(figsize=(7, max(3, .3 * len(imp))))
ax.barh(imp.index, imp.values, xerr=err.values, color="#2a6f97"); ax.set_xlabel("Drop in PR-AUC when shuffled")
ax.set_title(f"Permutation importance — {best}"); plt.tight_layout(); plt.show()

## 6. Hotspots: cluster crashes, then rank by cost vs. by count
Crashes are projected to metres and grouped with **DBSCAN** (density-based clustering: crashes within `EPS_METERS` of each other chain into one hotspot;
isolated crashes are noise). Each hotspot is then scored by **total crash cost** (severity-weighted) and by **crash count**.

In [ ]:
geo = df.dropna(subset=["lat", "lon"]).copy()
lat0 = geo.lat.mean()
XY = np.c_[geo.lon * 111_320 * np.cos(np.radians(lat0)), geo.lat * 110_540]   # equirectangular approx., fine at county scale

def build_hotspots(eps, min_n):
    lab = DBSCAN(eps=eps, min_samples=min_n, n_jobs=-1).fit_predict(XY)
    g = geo.assign(hotspot=lab); g = g[g.hotspot >= 0]
    h = g.groupby("hotspot").agg(
        crashes=("cost", "size"), total_cost=("cost", "sum"), severe=("severe", "sum"),
        fatal=("kabco", lambda s: (s == "K").sum()), lat=("lat", "mean"), lon=("lon", "mean"),
        town=("town", lambda s: s.mode().iat[0])).reset_index()
    h["cost_per_crash"] = h.total_cost / h.crashes
    h["rank_count"] = h.crashes.rank(ascending=False, method="min").astype(int)
    h["rank_cost"] = h.total_cost.rank(ascending=False, method="min").astype(int)
    h["rank_shift"] = h.rank_count - h.rank_cost            # >0: cost ranking moves it UP the list
    return h.sort_values("rank_cost"), lab

hot, labels = build_hotspots(EPS_METERS, MIN_CRASHES)
geo["hotspot"] = labels
print(f"{len(hot)} hotspots contain {(labels >= 0).mean():.1%} of geocoded crashes")
hot.head(15).assign(total_cost=lambda d: (d.total_cost / 1e6).round(1), cost_per_crash=lambda d: d.cost_per_crash.round(0),
                    lat=lambda d: d.lat.round(4), lon=lambda d: d.lon.round(4)).rename(columns={"total_cost": "total_cost_$M"})

In [ ]:
def compare_rankings(h, n):
    top_c = set(h.nsmallest(n, "rank_count").hotspot); top_k = set(h.nsmallest(n, "rank_cost").hotspot)
    return len(top_c & top_k), len(top_c & top_k) / len(top_c | top_k)

rho, _ = spearmanr(hot.rank_count, hot.rank_cost)
print(f"Spearman correlation between count rank and cost rank: {rho:.3f}")
for n in sorted({10, TOP_N, 50}):
    if n <= len(hot):
        k, j = compare_rankings(hot, n); print(f"Top-{n}: {k} hotspots in common ({n-k} differ);  Jaccard = {j:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sc = ax[0].scatter(hot.crashes, hot.total_cost / 1e6, c=hot.severe, cmap="viridis", s=30)
ax[0].set_xlabel("Crash count"); ax[0].set_ylabel("Total crash cost ($M)"); ax[0].set_title("Count vs. cost per hotspot")
plt.colorbar(sc, ax=ax[0], label="# severe (K+A)")
ax[1].scatter(hot.rank_count, hot.rank_cost, s=18, c="#2a6f97"); ax[1].plot([1, len(hot)], [1, len(hot)], "k--", lw=1)
ax[1].set_xlabel("Rank by count"); ax[1].set_ylabel("Rank by cost"); ax[1].set_title("Rank comparison (below line = cost ranks it higher)")
plt.tight_layout(); plt.show()

print(f"\n'Hidden' hotspots: top-{TOP_N} by COST but NOT top-{TOP_N} by COUNT (what the count-only approach misses)")
hid = hot[(hot.rank_cost <= TOP_N) & (hot.rank_count > TOP_N)]
hid[["hotspot", "town", "crashes", "severe", "fatal", "total_cost", "rank_count", "rank_cost"]].assign(total_cost=lambda d: (d.total_cost / 1e6).round(1)).rename(columns={"total_cost": "total_cost_$M"})

In [ ]:
# Robustness: do conclusions depend on the clustering parameters?
rob = []
for eps in [50, 100, 150, 250]:
    for mn in [10, 15, 25]:
        h, _ = build_hotspots(eps, mn)
        if len(h) >= 5:
            n = min(TOP_N, len(h)); k, j = compare_rankings(h, n)
            rob.append({"eps_m": eps, "min_crashes": mn, "hotspots": len(h), f"top{n}_overlap": k,
                        "spearman": spearmanr(h.rank_count, h.rank_cost)[0]})
pd.DataFrame(rob).round(3)

## 7. What do the costliest hotspots have in common?
Compare crash characteristics inside the top-cost hotspots against all other crashes. **Lift** > 1 means the condition is over-represented in costly hotspots.

In [ ]:
top_ids = set(hot.nsmallest(TOP_N, "rank_cost").hotspot)
geo["in_top"] = geo.hotspot.isin(top_ids)
print(f"{geo.in_top.sum():,} crashes in top-{TOP_N} cost hotspots vs {(~geo.in_top).sum():,} elsewhere")
rows = []
for c in cat_feats + ["rush_hour", "late_night", "is_weekend"]:
    a = geo.loc[geo.in_top, c].astype(str).value_counts(normalize=True)
    b = geo.loc[~geo.in_top, c].astype(str).value_counts(normalize=True)
    cnt = geo.loc[geo.in_top, c].astype(str).value_counts()
    for v in a.index:
        if cnt[v] >= 30:
            rows.append({"feature": c, "value": v, "share_in_top": a[v], "share_elsewhere": b.get(v, 0.0),
                         "lift": a[v] / max(b.get(v, 0.0), 1e-9), "n_top": cnt[v]})
cmp_ = pd.DataFrame(rows).sort_values("lift", ascending=False)
display(cmp_.head(12).round(3)); display(cmp_.tail(6).round(3))

## 8. Map and exports

In [ ]:
os.makedirs("outputs", exist_ok=True)
hot.to_csv("outputs/hotspots_ranked.csv", index=False)
res.to_csv("outputs/model_metrics.csv")
cmp_.to_csv("outputs/top_hotspot_feature_lift.csv", index=False)

try:
    import folium
    top = hot.nsmallest(TOP_N, "rank_cost")
    m = folium.Map(location=[geo.lat.mean(), geo.lon.mean()], zoom_start=10, tiles="cartodbpositron")
    maxc = top.total_cost.max()
    for _, r in top.iterrows():
        in_count_top = r.rank_count <= TOP_N
        folium.CircleMarker([r.lat, r.lon], radius=6 + 22 * (r.total_cost / maxc) ** .5,
            color="#b23a48" if not in_count_top else "#2a6f97", fill=True, fill_opacity=.6, weight=1.5,
            tooltip=(f"{r.town} | cost rank {int(r.rank_cost)} / count rank {int(r.rank_count)} | "
                     f"{int(r.crashes)} crashes, {int(r.severe)} severe, {int(r.fatal)} fatal | ${r.total_cost/1e6:.1f}M")).add_to(m)
    legend = ('<div style="position:fixed;bottom:20px;left:20px;z-index:9999;background:white;padding:8px 10px;'
              'border:1px solid #999;font-size:12px"><b>Top cost hotspots</b><br>'
              '<span style="color:#2a6f97">●</span> also top by count<br>'
              '<span style="color:#b23a48">●</span> missed by count ranking<br>size = total crash cost</div>')
    m.get_root().html.add_child(folium.Element(legend))
    m.save("outputs/hotspot_map.html"); print("Saved outputs/hotspot_map.html")
    display(m)
except ImportError:
    fig, ax = plt.subplots(figsize=(7, 7)); top = hot.nsmallest(TOP_N, "rank_cost")
    ax.scatter(geo.lon, geo.lat, s=1, c="lightgrey")
    ax.scatter(top.lon, top.lat, s=20 + 400 * top.total_cost / top.total_cost.max(), c="#b23a48", alpha=.6)
    plt.show(); print("pip install folium for an interactive map")

if USING_SYNTHETIC: print("\n*** REMINDER: every number above came from FAKE data. ***")

## Notes, limitations & next steps
- **Cost weights** use FHWA comprehensive per-injury-level costs (K/A/B/C/O). Confirm the numbers/year you cite; the FHWA Crash Cost tool (mentioned in the proposal as future work) can replace `CRASH_COST`.
- **One fatal crash can dominate a hotspot's total cost** ($11M vs. $12K for PDO). Consider also reporting the K+A count, cost excluding fatals, or an empirical-Bayes adjusted ranking so rankings are not driven by a single random event.
- **Exposure** (traffic volume/AADT) is not in the crash data; a high count may just mean a busy road. CTDOT AADT data would let you compute crash *rates*.
- **Predictor scope:** the model predicts severity *given a crash occurred* — it identifies conditions associated with severity, not causes.
- The proposal's idea of weighting crashes by contributing factors / distraction can be added by multiplying each crash's `cost` by a factor weight in section 3.